# 02 — Bronze Data Quality & Reconciliation
**Validate the real-time bronze layer and reconcile with Eventhouse**

## Data Flow (Actual)
```
Eventstream (RTIbikeRental)
  ├── Destination 1 → Bronze Lakehouse (bikeraw_tb)   ← snapshot data
  └── Destination 2 → Eventhouse (BikeRentalEH)       ← real-time KQL
```

## What This Notebook Does
1. **Data quality audit** on `bikeraw_tb` — the real bronze table populated by Eventstream
2. **Station profile analysis** — unique stations, neighbourhoods, capacity distribution
3. **Freshness check** — detect stale data (Eventstream may be paused/inactive)
4. **Station reference table** — create `bicycle_station_ref` for downstream Silver/Gold
5. **Eventhouse reconciliation** — compare bronze row counts with KQL DB

### Prerequisites
- Attach **bikerental_bronze_raw** as the default lakehouse
- Eventstream `RTIbikeRental` should have run at least once

In [ ]:
# ============================================================
# CELL 1 — Configuration
# ============================================================

BRONZE_LAKEHOUSE   = "bikerental_bronze_raw"
RAW_TABLE          = "bikeraw_tb"              # Real bronze table from Eventstream
EVENTHOUSE_NAME    = "bikerentaleventhouse"
KQL_DB_NAME        = "BikeRentalEH"            # Eventhouse KQL database

# Expected schema (from BicyclesRental sample via Eventstream)
EXPECTED_COLUMNS = {
    "BikepointID", "Street", "Neighbourhood",
    "Latitude", "Longitude",
    "No_Bikes", "No_Empty_Docks",
}

print(f"Bronze Lakehouse : {BRONZE_LAKEHOUSE}")
print(f"Source table     : {RAW_TABLE}")
print(f"Eventhouse       : {EVENTHOUSE_NAME} / {KQL_DB_NAME}")

In [ ]:
# ============================================================
# CELL 2 — Data Quality Audit on bikeraw_tb
# ============================================================

from pyspark.sql.functions import (
    col, count, lit, when, isnull, isnan,
    min as spark_min, max as spark_max, avg as spark_avg,
    sum as spark_sum, countDistinct, current_timestamp,
)

df_raw = spark.read.format("delta").table(RAW_TABLE)
total_rows = df_raw.count()

print(f"Table: {RAW_TABLE}")
print(f"Total rows: {total_rows:,}\n")

# ── Schema validation ─────────────────────────────────────────
actual_cols = set(df_raw.columns)
missing = EXPECTED_COLUMNS - actual_cols
extra   = actual_cols - EXPECTED_COLUMNS

if missing:
    print(f"⚠ Missing columns: {missing}")
else:
    print("✓ All expected columns present")
if extra:
    print(f"ℹ Extra columns (from Eventstream): {extra}")

# ── Null / quality report ─────────────────────────────────────
print("\n── Column Quality Report ────────────────────────────")
for c in sorted(df_raw.columns):
    null_count = df_raw.filter(col(c).isNull()).count()
    null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0
    status = "✓" if null_pct == 0 else "⚠"
    print(f"  {status} {c:<25} nulls: {null_count:>5}  ({null_pct:.1f} %)")

# ── Range checks for numeric columns ─────────────────────────
print("\n── Numeric Range Checks ─────────────────────────────")
for c in ["No_Bikes", "No_Empty_Docks", "Latitude", "Longitude"]:
    if c in actual_cols:
        stats = df_raw.agg(
            spark_min(c).alias("min"),
            spark_max(c).alias("max"),
            spark_avg(c).alias("avg"),
        ).collect()[0]
        neg = df_raw.filter(col(c) < 0).count()
        print(f"  {c:<20} min={stats['min']:<10} max={stats['max']:<10} avg={stats['avg']:.2f}  negatives={neg}")

# ── Duplicate check ───────────────────────────────────────────
unique_stations = df_raw.select("BikepointID").distinct().count()
dup_rows = total_rows - unique_stations
print(f"\n── Uniqueness ───────────────────────────────────────")
print(f"  Unique BikepointIDs : {unique_stations:,}")
print(f"  Duplicate rows      : {dup_rows:,}")
if dup_rows > 0:
    print("  ⚠ Duplicates likely from multiple Eventstream snapshots")

In [ ]:
# ============================================================
# CELL 3 — Neighbourhood Analysis & Freshness Check
# ============================================================

from pyspark.sql.functions import hour, dayofweek, date_format

# ── Neighbourhood breakdown ───────────────────────────────────
print("── Neighbourhood Distribution ───────────────────────")
df_hood_summary = (
    df_raw
    .groupBy("Neighbourhood")
    .agg(
        count("*").alias("stations"),
        spark_sum("No_Bikes").alias("total_bikes"),
        spark_sum("No_Empty_Docks").alias("total_empty_docks"),
        spark_sum(col("No_Bikes") + col("No_Empty_Docks")).alias("total_capacity"),
        spark_avg(
            col("No_Bikes") / (col("No_Bikes") + col("No_Empty_Docks"))
        ).alias("avg_utilization"),
    )
    .orderBy("stations", ascending=False)
)
df_hood_summary.show(30, truncate=False)

# ── Freshness check (if Eventstream adds timestamp columns) ───
ts_candidates = [c for c in df_raw.columns if "time" in c.lower() or "date" in c.lower() or "ts" in c.lower()]
if ts_candidates:
    print(f"── Freshness Check (column: {ts_candidates[0]}) ─────────────")
    ts_col = ts_candidates[0]
    freshness = df_raw.agg(
        spark_min(ts_col).alias("earliest"),
        spark_max(ts_col).alias("latest"),
    ).collect()[0]
    print(f"  Earliest : {freshness['earliest']}")
    print(f"  Latest   : {freshness['latest']}")
else:
    print("── Freshness Check ──────────────────────────────────")
    print("  ℹ No timestamp column detected in bikeraw_tb.")
    print("  BicyclesRental sample data is a point-in-time snapshot.")
    print("  Freshness depends on when Eventstream last ran.")

In [ ]:
# ============================================================
# CELL 4 — Create Station Reference Table
# ============================================================
# Builds a clean, deduplicated station reference from bikeraw_tb.
# Downstream NB03 Silver and NB04 Gold read from this.

from pyspark.sql.functions import lit, col, coalesce

df_station_ref = (
    df_raw
    .select(
        col("BikepointID"),
        col("Street"),
        col("Neighbourhood"),
        col("Latitude"),
        col("Longitude"),
        col("No_Bikes"),
        col("No_Empty_Docks"),
        (col("No_Bikes") + col("No_Empty_Docks")).alias("total_docks"),
    )
    .distinct()
)

ref_count = df_station_ref.count()
print(f"Station reference: {ref_count:,} unique stations")

# Write as a separate Bronze table
df_station_ref.write.format("delta").mode("overwrite").saveAsTable(
    "bicycle_station_ref"
)
print("✓ Written → bicycle_station_ref")

# Verify
verify = spark.read.format("delta").table("bicycle_station_ref").count()
print(f"  Verified: {verify:,} rows in bicycle_station_ref")

# Show top neighbourhoods by capacity
print("\n── Top Neighbourhoods by Capacity ───────────────────")
df_station_ref.groupBy("Neighbourhood").agg(
    count("*").alias("stations"),
    spark_sum("total_docks").alias("total_capacity"),
    spark_avg("total_docks").alias("avg_docks_per_station"),
).orderBy("total_capacity", ascending=False).show(20, truncate=False)

In [ ]:
# ============================================================
# CELL 5 — Bronze Summary & Eventhouse Reconciliation Guide
# ============================================================

total_rows = df_raw.count()
ref_rows   = spark.read.format("delta").table("bicycle_station_ref").count()
hood_count = df_hood_summary.count()

print("=" * 60)
print("  BRONZE LAYER — SUMMARY")
print("=" * 60)
print(f"  bikeraw_tb          : {total_rows:,} rows")
print(f"  bicycle_station_ref : {ref_rows:,} unique stations")
print(f"  Neighbourhoods      : {hood_count:,}")
print("=" * 60)

# ── Eventhouse reconciliation guide ───────────────────────────
print("\n── Eventhouse Reconciliation ─────────────────────────")
print("  The Eventstream RTIbikeRental sends data to BOTH:")
print("    1. Lakehouse → bikerental_bronze_raw / bikeraw_tb  (batch/Delta)")
print("    2. Eventhouse → BikeRentalEH                       (real-time/KQL)")
print()
print("  To compare row counts, run this KQL query in the Eventhouse:")
print("  ┌──────────────────────────────────────────────────┐")
print("  │  bikeraw_tb | count                              │")
print("  └──────────────────────────────────────────────────┘")
print(f"  Expected: ~{total_rows:,} rows (should match lakehouse)")
print()
print("  If counts differ, the Eventstream may have been paused")
print("  for one destination but not the other. Check the")
print("  Eventstream diagram — both destinations should show 'Active'.")

print("\n✅ NB02 complete — Bronze validated, station_ref created.")
print("   Next: run NB03 (Silver Enrich & Transform)")